In [1]:
from pathlib import Path

data_dir = Path("data")
separator = "<|endoftext|>"

splits = {}

for name in ["train", "valid", "test"]:
    path = data_dir / f"{name}.txt"
    text = path.read_text(encoding="utf-8")
    stories = [story.strip() for story in text.split(separator) if story.strip()]
    splits[name] = stories

    print(f"{name}: {len(stories):,} stories, {path.stat().st_size / 1_000_000:.1f} MB")
    print("First story:")
    print(stories[0][:500])
    print("-" * 50)

train: 100,000 stories, 91.2 MB
First story:
One hot day, Robert and his dad decided to go to the park. They brought their ball with them and once they arrived at the park, Robert wanted to play catch. 

Robert's dad said, "Let's play next to the fence. Be careful and don't throw the ball too far."

Robert played catch for a few minutes and threw the ball very high! But he missed it and the ball went over the fence. 

Robert's dad said, "Oh no, the ball went over the fence. I'll try to catch it." 

Robert's dad ran to the fence and grabbed
--------------------------------------------------
valid: 2,000 stories, 1.9 MB
First story:
Once upon a time there was a little girl named Nina. She was only three years old but she was very excited to see a model she had made. She had used paper and glue to make it, and it was very thin.

Nina couldn't wait to show her parents the model. She ran into the living room and saw them sitting there.

"Mummy, Daddy, look!" said Nina, proudly. "I've made a

In [2]:
train = set(splits["train"])
valid = set(splits["valid"])
test = set(splits["test"])

print("Train/validation duplicates:", len(train & valid))
print("Train/test duplicates:", len(train & test))
print("Validation/test duplicates:", len(valid & test))

Train/validation duplicates: 55
Train/test duplicates: 77
Validation/test duplicates: 6


In [3]:
# Keep the test set unchanged.
test_stories = set(splits["test"])

# Remove test stories from validation.
clean_valid = [
    story for story in splits["valid"]
    if story not in test_stories
]

# Remove validation and test stories from training.
held_out = set(clean_valid) | test_stories
clean_train = [
    story for story in splits["train"]
    if story not in held_out
]

print(f"Training:   {len(splits['train']):,} → {len(clean_train):,}")
print(f"Validation: {len(splits['valid']):,} → {len(clean_valid):,}")
print(f"Test:       {len(splits['test']):,} (unchanged)")

print("Remaining train/validation overlap:", len(set(clean_train) & set(clean_valid)))
print("Remaining train/test overlap:", len(set(clean_train) & test_stories))
print("Remaining validation/test overlap:", len(set(clean_valid) & test_stories))

Training:   100,000 → 99,868
Validation: 2,000 → 1,994
Test:       2,000 (unchanged)
Remaining train/validation overlap: 0
Remaining train/test overlap: 0
Remaining validation/test overlap: 0


In [4]:
clean_dir = Path("data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

clean_splits = {
    "train": clean_train,
    "valid": clean_valid,
    "test": splits["test"],
}

for name, stories in clean_splits.items():
    output = clean_dir / f"{name}.txt"
    output.write_text(
        "\n<|endoftext|>\n".join(stories) + "\n<|endoftext|>\n",
        encoding="utf-8",
    )
    print(f"Saved {output}")

Saved data/clean/train.txt
Saved data/clean/valid.txt
Saved data/clean/test.txt
